# Data Processing and Preparation for Power BI

## Goal
The purpose of this notebook is to process the raw JSON files downloaded from the Google Analytics API. The script transforms the raw data into a single, clean, and analysis-ready dataset suitable for visualization in Power BI.

---

## Process
The notebook performs the following key steps in the **Transform** and **Load** stages of an ETL process:

1.  **Load Data**: Iterates through all JSON files.
2.  **Initial Transformation**: For each file, the script:
    * Creates a pandas DataFrame.
    * Renames columns.
    * Converts all columns to their appropriate data types.
3.  **Feature Engineering**:
    * Creates a primary `date` column (datetime type) from the original `year` and `month` columns.
    * Calculates two new key performance indicators (KPIs):
        * `engagement_rate`: To measure the quality of user sessions.
        * `new_users_ratio`: To measure audience growth.
4.  **Data Aggregation**:
    * Extracts the source website name from each filename and adds it as a `website` column.
    * Combines all the individual DataFrames into a single DataFrame.
5.  **Final Output**:
    * Saves the master dataset as a **CSV** file.

**Required libraries**

In [1]:
import pandas as pd
import json
import os
import re
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Data processing**

*1) Functions definition*

In [3]:
def make_df(data):
    """Convert data to DataFrame and preprocess it."""
    df = pd.DataFrame(data)

    # 1. Columns rename
    df.rename(columns={
    'activeUsers': 'active_users',
    'newUsers': 'new_users',
    'sessionsPerUser': 'sessions_per_user',
    'screenPageViews': 'screen_page_views',
    'engagedSessions': 'engaged_sessions',
    'averageSessionDuration': 'average_session_duration',
    'userEngagementDuration': 'user_engagement_duration',
    'deviceCategory': 'device_category',
    'sessionDefaultChannelGroup': 'channel_group'
    }, inplace=True)

    # 2. Data type setting
    df['year'] = df['year'].astype(int)
    df['month'] = df['month'].astype(int)
    df['device_category'] = df['device_category'].astype(str)
    df['channel_group'] = df['channel_group'].astype(str)
    df['active_users'] = df['active_users'].astype(int)
    df['new_users'] = df['new_users'].astype(int)
    df['sessions'] = df['sessions'].astype(int)
    df['sessions_per_user'] = df['sessions_per_user'].astype(float)
    df['screen_page_views'] = df['screen_page_views'].astype(int)
    df['engaged_sessions'] = df['engaged_sessions'].astype(int)
    df['average_session_duration'] = df['average_session_duration'].astype(float)
    df['user_engagement_duration'] = df['user_engagement_duration'].astype(float)

    # Preparation a column with the full date
    date_column_data = pd.to_datetime(df[['year', 'month']].assign(day=1))
    df.insert(0, 'date', date_column_data)

    # Calculation of additional KPIs
    df["engagement_rate"] = np.divide(df["engaged_sessions"], df["sessions"], out=np.zeros_like(df["engaged_sessions"], dtype=float), where=df["sessions"]!=0)
    df["new_users_ratio"] = np.divide(df["new_users"], df["active_users"], out=np.zeros_like(df["new_users"], dtype=float), where=df["active_users"]!=0)

    return df

In [4]:
def filename_extraction(filename):
    """Extract the domain name with the extension (.cz, .sk) from the filename"""
    # Regular expression for extracting domain name
    pattern = r'([a-zA-Z0-9\.\-]+(?:\.cz|\.sk))-\d{4}-\d{2}-\d{2}-\d{4}-\d{2}-\d{2}\.json'

    # Using a regular expression
    match = re.search(pattern, filename)

    if match:
        website = match.group(1)
        return website

    else:
        print("Website not found in filename:", filename)
        return None

*2) Start of processing*

In [5]:
# Path to the data folder
FOLDER_PATH = '/content/drive/MyDrive/01_data_science/03_projects/02_API_GA/3_data/raw'
SAVE_DIR = '/content/drive/MyDrive/01_data_science/03_projects/02_API_GA/3_data/processed_data'

# List of files in the folder
files = os.listdir(FOLDER_PATH)

In [6]:
# Initialize empty list to store DataFrames
df_list = []

# Iteration through individual files
for filename in files:
    file_path = os.path.join(FOLDER_PATH, filename)
    with open(file_path, 'r', encoding='utf-8') as json_file:
        data = json.load(json_file)

        # Create a DataFrame from the data
        df = make_df(data)

        # Extract website from filename
        website = filename_extraction(filename)

        # # Add the extracted domain name to the DataFrame
        if website:
            df.insert(0, 'website', website)

        # Append the DataFrame to the list
        df_list.append(df)

# Concatenate all DataFrames into a single one
final_df = pd.concat(df_list, ignore_index=True)

# Saving processed data
csv_path = os.path.join(SAVE_DIR, 'ga_data_processed.csv')
final_df.to_csv(csv_path, index=False)